## Section 0 - Drive Mount,Code Import, HF Login , OUT/Cache Dir

In [1]:

# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Add Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir --> out_quick 
os.makedirs('/content/drive/MyDrive/indic_synth/out_quick', exist_ok=True)
OUT = '/content/drive/MyDrive/indic_synth/out_quick'

Mounted at /content/drive
/content
Cloning into 'synthetic-data-pipeline'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 231 (delta 70), reused 213 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 5.51 MiB | 3.50 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/synthetic-data-pipeline
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


In [2]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# Add Cache Dir for Gemma (24gb) - only Run cell if you have enough G-drive storage
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"

## Section 1 - 3 (till TTS Generation)

In [3]:
# Install for all stages till tts generation
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 125.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 98.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 56.7 MB/s eta 0:00:00


In [4]:
!python scripts/run.py --config config.quick.yaml --stages data_acquisition

05:16:53 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
05:16:53 INFO    run | =========== stage: data_acquisition ===========
05:16:53 INFO    run | Acquisition config: {'source': 'hf', 'repo_id': 'ai4bharat/Kathbath', 'local_dir': 'fake_kathbath', 'languages': ['hindi', 'malayalam', 'tamil'], 'split': 'valid', 'speakers_per_language': 2, 'clips_per_speaker': 2, 'min_total_speakers': 4, 'gender_balance': True, 'ref_min_dur': 3.0, 'ref_max_dur': 15.0, 'seed': 1234, 'out_dir': '/content/drive/MyDrive/indic_synth/out_quick', 'hf_token': True, 'max_retries': 4, 'retry_backoff': 2.0, 'force_catalog': False}
05:16:54 INFO    run | [hindi] cataloging 2 parquet file(s) (audio column skipped)
05:17:32 INFO    run | [hindi] valid-00000-of-00002.parquet -> 1576 rows
05:18:11 INFO    run | [hindi] valid-00001-of-00002.parquet -> 1575 rows
05:18:11 INFO    run | [malayalam] cataloging 2 parquet file(s) (audio column skipped)
05:18:31 INFO    run | [ma

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages audio_engineering

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages sentence_generation

### Section 4 - TTS Generation

In [ ]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml

In [ ]:
# Restart The session and Run Section 0 

In [ ]:
# Check transformer Version  # -> 4.49.0
import transformers
print(transformers.__version__)

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages tts_generation

## Section - 5 QC 

In [ ]:
# Update the Versions back
!pip install -r requirements.txt

In [ ]:
# Restart and Run Section zero

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages quality_control

In [ ]:
print(open(f'{OUT}/qc_summary.json').read())

### Sanity Check - Rejected by QC Review

In [ ]:
import json
import os
from typing import Dict, Iterable, Iterator, List
def read_jsonl(path: str) -> List[Dict]:
    """Read a JSONL file into a list of dicts. Missing file -> empty list."""
    if not os.path.exists(path):
        return []
    rows: List[Dict] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [ ]:
# sanity-check thresholds: listen to a couple of QC failures
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:] # Edit show much you want to Verify 
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))